In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import rankings

# 1. Feature Engineering: Lags and Windows
def create_forecasting_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values('date', inplace=True)
    
    # Calendar features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    
    # Target Lags (Crucial for XGBoost time-series)
    df['lag_1'] = df['trucks_needed'].shift(1)
    df['lag_7'] = df['trucks_needed'].shift(7)
    
    # Rolling Statistics
    df['rolling_mean_7'] = df['trucks_needed'].shift(1).rolling(window=7).mean()
    df['rolling_std_7'] = df['trucks_needed'].shift(1).rolling(window=7).std()
    
    # Drop rows with NaN values caused by shifting
    df.dropna(inplace=True)
    return df

# Load and process data
df_raw = pd.read_csv('./content/advanced_logistics_data.csv')
df_feat = create_forecasting_features(df_raw)

# 2. Split Features and Target
feature_cols = ['precipitation_mm', 'is_holiday', 'day_of_week', 'month', 
                'day_of_year', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7']

X = df_feat[feature_cols]
y = df_feat['trucks_needed']

# 3. TimeSeries Cross-Validation Setup
tscv = TimeSeriesSplit(n_splits=5)
for train_idx, val_idx in tscv.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# 4. Hyperparameter-Tuned XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.03,      # Low learning rate prevents overfitting to outliers
    max_depth=4,             # Shallow trees keep model generalized
    subsample=0.8,           # Row subsampling adds robustness against noise
    colsample_bytree=0.8,    # Column subsampling
    objective='reg:squarederror',
    random_state=42
)

# Train model
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"XGBoost Training Complete. Final Val RMSE: {np.sqrt(mean_squared_error(y_val, xgb_model.predict(X_val))):.2f}")


XGBoost Training Complete. Final Val RMSE: 0.35


In [7]:
# 1. Define the future 15-day timeline starting from the last date in your dataset
future_dates = pd.date_range(start=df_feat['date'].max() + pd.Timedelta(days=1), periods=15, freq='D')

# 2. Input future conditions (e.g., a heavy rainstorm on Day 5, no holidays)
future_precip = [0.0]*4 + [35.0] + [0.0]*10  
future_holidays = [0]*15                     

forecast_results = []
history_trucks = list(df_feat['trucks_needed'].values)

# 3. Step-by-step recursive prediction loop
for i, current_date in enumerate(future_dates):
    lag_1 = history_trucks[-1]
    lag_7 = history_trucks[-7]
    rolling_mean_7 = np.mean(history_trucks[-7:])
    rolling_std_7 = np.std(history_trucks[-7:])
    
    # Match the exact feature columns used during xgb_model.fit()
    input_features = pd.DataFrame([{
        'precipitation_mm': future_precip[i],
        'is_holiday': future_holidays[i],
        'day_of_week': current_date.dayofweek,
        'month': current_date.month,
        'day_of_year': current_date.dayofyear,
        'lag_1': lag_1,
        'lag_7': lag_7,
        'rolling_mean_7': rolling_mean_7,
        'rolling_std_7': rolling_std_7
    }])[feature_cols] # Ensures correct column ordering
    
    # Generate prediction
    pred_trucks = max(1, int(np.round(xgb_model.predict(input_features)[0])))
    pred_staff = max(1, int(np.round(pred_trucks * 2.3)))
    
    # Feed back into history for next day's lag calculation
    history_trucks.append(pred_trucks)
    
    forecast_results.append({
        'Date': current_date.strftime('%Y-%m-%d'),
        'Day': current_date.day_name(),
        'Rainfall_mm': future_precip[i],
        'Predicted_Trucks': pred_trucks,
        'Predicted_Staff': pred_staff
    })

# 4. Display the response dataframe
schedule_df = pd.DataFrame(forecast_results)
schedule_df


,Date,Day,Rainfall_mm,Predicted_Trucks,Predicted_Staff
0,2025-01-16,Thursday,0.0,36,83
1,2025-01-17,Friday,0.0,23,53
2,2025-01-18,Saturday,0.0,19,44
3,2025-01-19,Sunday,0.0,17,39
4,2025-01-20,Monday,35.0,36,83
5,2025-01-21,Tuesday,0.0,37,85
6,2025-01-22,Wednesday,0.0,37,85
7,2025-01-23,Thursday,0.0,36,83
8,2025-01-24,Friday,0.0,23,53
9,2025-01-25,Saturday,0.0,20,46
